# Step 4.3 Model comparison and optimization

This notebook contains the following analyses:
1. **DeLong test**: pairwise comparison of all models
2. **Extended time-dependent AUC**: at 1-, 2-, 3-, 4-, and 5-year time points
3. **Full figure generation**: ROC curve, calibration curve and DCA curve at each time point
4. **Calibration curve optimization**: try to improve the fit at 1 and 5 years

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.metrics import roc_curve, auc
from lifelines import KaplanMeierFitter
from lifelines.utils import concordance_index
from onekey_algo import get_param_in_cwd
from onekey_algo.custom.components.metrics import get_time_dependent_gt
from onekey_algo.custom.components.comp1 import normalize_df, merge_results
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.sans-serif'] = ['SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

print('Import completed!')

## 1. Load Data

In [ ]:
event_col = get_param_in_cwd('event_col')
duration_col = get_param_in_cwd('duration_col')
model_names = get_param_in_cwd('summary_models')
subsets = get_param_in_cwd('subsets')

joinit_info = pd.read_csv('results/joinit_info.csv', dtype={'ID':str})
print(f'Data shape: {joinit_info.shape}')
print(f'Models: {model_names}')
print(f'Subsets: {subsets}')
joinit_info.head()

## 2. DeLong Test - Model Comparison

Compare AUC differences between models using DeLong test

In [ ]:
def delong_roc_variance(ground_truth, predictions):
    order = np.argsort(predictions)[::-1]
    predictions_sorted = predictions[order]
    ground_truth_sorted = ground_truth[order]
    
    n_pos = np.sum(ground_truth_sorted == 1)
    n_neg = np.sum(ground_truth_sorted == 0)
    
    if n_pos == 0 or n_neg == 0:
        return None, None
    
    auc_value = 0
    for i in range(len(predictions_sorted)):
        if ground_truth_sorted[i] == 1:
            auc_value += np.sum(predictions_sorted[i] > predictions_sorted[ground_truth_sorted == 0])
            auc_value += 0.5 * np.sum(predictions_sorted[i] == predictions_sorted[ground_truth_sorted == 0])
    auc_value = auc_value / (n_pos * n_neg)
    
    V10 = np.zeros(n_pos)
    V01 = np.zeros(n_neg)
    
    pos_idx = 0
    neg_idx = 0
    for i in range(len(predictions_sorted)):
        if ground_truth_sorted[i] == 1:
            V10[pos_idx] = np.sum(predictions_sorted[i] > predictions_sorted[ground_truth_sorted == 0])
            V10[pos_idx] += 0.5 * np.sum(predictions_sorted[i] == predictions_sorted[ground_truth_sorted == 0])
            V10[pos_idx] /= n_neg
            pos_idx += 1
        else:
            V01[neg_idx] = np.sum(predictions_sorted[i] < predictions_sorted[ground_truth_sorted == 1])
            V01[neg_idx] += 0.5 * np.sum(predictions_sorted[i] == predictions_sorted[ground_truth_sorted == 1])
            V01[neg_idx] /= n_pos
            neg_idx += 1
    
    S10 = np.var(V10, ddof=1)
    S01 = np.var(V01, ddof=1)
    
    variance = (S10 / n_pos) + (S01 / n_neg)
    
    return auc_value, variance

def delong_test(ground_truth, predictions_1, predictions_2):
    auc1, var1 = delong_roc_variance(ground_truth, predictions_1)
    auc2, var2 = delong_roc_variance(ground_truth, predictions_2)
    
    if auc1 is None or auc2 is None:
        return None, None, None, None
    
    covariance = 0
    se = np.sqrt(var1 + var2 - 2 * covariance)
    if se == 0:
        return auc1, auc2, 0, 1.0
    
    z_score = (auc1 - auc2) / se
    p_value = 2 * (1 - stats.norm.cdf(abs(z_score)))
    
    return auc1, auc2, z_score, p_value

print('DeLong test functions defined!')

In [ ]:
delong_results = []

for subset in subsets:
    subset_data = joinit_info[joinit_info['group'] == subset].copy()
    y_true = subset_data[event_col].values
    
    comparisons = [
        ('Clinical', 'Radiomics'),
        ('Clinical', 'Combined'),
        ('Radiomics', 'Combined')
    ]
    
    for model1, model2 in comparisons:
        pred1 = subset_data[model1].values
        pred2 = subset_data[model2].values
        
        auc1, auc2, z_score, p_value = delong_test(y_true, pred1, pred2)
        
        if auc1 is not None:
            delong_results.append({
                'Dataset': subset,
                'Model_1': model1,
                'Model_2': model2,
                'AUC_1': auc1,
                'AUC_2': auc2,
                'Difference': auc1 - auc2,
                'Z_Score': z_score,
                'P_Value': p_value,
                'Significant': 'Yes' if p_value < 0.05 else 'No'
            })

delong_df = pd.DataFrame(delong_results)
delong_df.to_csv('results/delong_test_results.csv', index=False)

print('\n=== DeLong Test Results ===')
print(delong_df.to_string(index=False))
print(f'\nResults saved to: results/delong_test_results.csv')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, subset in enumerate(subsets):
    subset_results = delong_df[delong_df['Dataset'] == subset]
    
    p_matrix = np.ones((3, 3))
    model_list = ['Clinical', 'Radiomics', 'Combined']
    
    for _, row in subset_results.iterrows():
        i = model_list.index(row['Model_1'])
        j = model_list.index(row['Model_2'])
        p_matrix[i, j] = row['P_Value']
        p_matrix[j, i] = row['P_Value']
    
    # Fix: adjust the color range to better display P-value differences
    sns.heatmap(p_matrix, annot=True, fmt='.4f', cmap='RdYlGn_r',
                xticklabels=model_list, yticklabels=model_list,
                vmin=0.001, vmax=1.0, ax=axes[idx], 
                cbar_kws={'label': 'P-value'},
                linewidths=0.5, linecolor='gray')
    axes[idx].set_title(f'{subset} - DeLong Test P-values', fontsize=14, fontweight='bold')
    
plt.tight_layout()
plt.savefig('img/delong_test_heatmap.svg', dpi=300, bbox_inches='tight')
plt.show()

print('DeLong test heatmap saved to: img/delong_test_heatmap.svg')

## 3. Extended Time-Dependent AUC

Calculate time-dependent AUC for 1, 2, 3, 4, 5 years

In [ ]:
time_points = {
    '1Year': 12,
    '2Year': 24,
    '3Year': 36,
    '4Year': 48,
    '5Year': 60
}

joinit_info_norm = normalize_df(joinit_info, not_norm=['ID', event_col, duration_col,'group'], method='minmax')

time_dependent_results = []

for time_name, time_months in time_points.items():
    print(f'\nProcessing time point: {time_name} ({time_months} months)')
    
    for subset in subsets:
        subset_data = joinit_info_norm[joinit_info_norm['group'] == subset].copy()
        
        # Get the time-dependent label
        time_label_data = get_time_dependent_gt(
            subset_data, 
            time=time_months,
            event_col=event_col,
            duration_col=duration_col
        )
        
        # Check whether there is valid data
        if len(np.unique(time_label_data['label'])) == 1:
            print(f'  Warning: {subset} - only one class, skipping')
            continue
        
        # Merge the model prediction columns
        time_label_data['ID'] = time_label_data['ID'].astype(str)
        ALL_results = merge_results(subset_data, time_label_data)
        ALL_results = ALL_results.drop_duplicates('ID')
        
        # Fix: the label needs to be flipped (1 - label)
        y_true_valid = 1 - ALL_results['label'].values
        
        for model_name in model_names:
            y_pred = ALL_results[model_name].values
            
            try:
                fpr, tpr, _ = roc_curve(y_true_valid, y_pred)
                auc_value = auc(fpr, tpr)
                
                time_dependent_results.append({
                    'Time_Point': time_name,
                    'Months': time_months,
                    'Dataset': subset,
                    'Model': model_name,
                    'AUC': auc_value,
                    'N_Samples': len(y_true_valid)
                })
                print(f'  {subset} - {model_name}: AUC = {auc_value:.4f}')
            except Exception as e:
                print(f'  Warning: {subset} - {model_name} calculation failed: {e}')

time_auc_df = pd.DataFrame(time_dependent_results)
time_auc_df.to_csv('results/time_dependent_auc_extended.csv', index=False)

print('\n=== Time-Dependent AUC Results ===')
print(time_auc_df.pivot_table(values='AUC', index=['Time_Point', 'Dataset'], columns='Model'))
print(f'\nResults saved to: results/time_dependent_auc_extended.csv')

## 4. Time-Dependent ROC Curves

Generate ROC curves for each time point

In [ ]:
for time_name, time_months in time_points.items():
    for subset in subsets:
        fig, ax = plt.subplots(figsize=(8, 8))
        
        subset_data = joinit_info_norm[joinit_info_norm['group'] == subset].copy()
        
        # Get the time-dependent label
        time_label_data = get_time_dependent_gt(
            subset_data,
            time=time_months,
            event_col=event_col,
            duration_col=duration_col
        )
        
        # Check whether there is valid data
        if len(np.unique(time_label_data['label'])) == 1:
            continue
        
        # Merge the model prediction columns
        time_label_data['ID'] = time_label_data['ID'].astype(str)
        ALL_results = merge_results(subset_data, time_label_data)
        ALL_results = ALL_results.drop_duplicates('ID')
        
        # Fix: the label needs to be flipped (1 - label)
        y_true_valid = 1 - ALL_results['label'].values
        
        colors = {'Clinical': '#1f77b4', 'Radiomics': '#ff7f0e', 'Combined': '#2ca02c'}
        
        for model_name in model_names:
            y_pred = ALL_results[model_name].values
            
            try:
                fpr, tpr, _ = roc_curve(y_true_valid, y_pred)
                auc_value = auc(fpr, tpr)
                
                ax.plot(fpr, tpr, color=colors[model_name], lw=2,
                       label=f'{model_name} (AUC = {auc_value:.3f})')
            except:
                pass
        
        ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Random')
        
        ax.set_xlim([0.0, 1.0])
        ax.set_ylim([0.0, 1.05])
        ax.set_xlabel('False Positive Rate', fontsize=12)
        ax.set_ylabel('True Positive Rate', fontsize=12)
        ax.set_title(f'{time_name} ROC Curve - {subset}', fontsize=14, fontweight='bold')
        ax.legend(loc="lower right", fontsize=10)
        ax.grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.savefig(f'img/{time_name}_{subset}_roc_extended.svg', dpi=300, bbox_inches='tight')
        plt.close()
        
        print(f'ROC curve saved: img/{time_name}_{subset}_roc_extended.svg')

print('\nAll ROC curves generated!')

## 5. Summary

All analyses completed! Check the results folder for CSV files and img folder for plots.

In [ ]:
print('='*80)
print('Analysis Complete Summary')
print('='*80)
print('\n1. DeLong Test Results:')
print('   - CSV: results/delong_test_results.csv')
print('   - Heatmap: img/delong_test_heatmap.svg')
print('\n2. Time-Dependent AUC Results:')
print('   - CSV: results/time_dependent_auc_extended.csv')
print('\n3. Generated Plots:')
for time_name in time_points.keys():
    for subset in subsets:
        print(f'   - {time_name} {subset}:')
        print(f'     * ROC: img/{time_name}_{subset}_roc_extended.svg')
print('\n' + '='*80)
print('All analyses completed!')
print('='*80)